In [1]:
!sudo apt-get update
!sudo apt-get install cuda-12-4

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,381 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,241 kB]
Get:13 https://r2u.stat.illinois.edu/u

In [2]:
!rm /usr/local/cuda
!ln -s /usr/local/cuda-12.4 /usr/local/cuda

In [3]:
!nvcc --version
!pip install nvcc4jupyter

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Mar_28_02:18:24_PDT_2024
Cuda compilation tools, release 12.4, V12.4.131
Build cuda_12.4.r12.4/compiler.34097967_0


In [4]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpc4uh6s4b".


In [5]:
!apt-get -y install postgresql postgresql-contrib
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres createdb testdb
!pip install psycopg2-binary

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libtypes-serialiser-perl logrotate netbase
  postgresql-14 postgresql-client-14 postgresql-client-common postgresql-common ssl-cert sysstat
Suggested packages:
  bsd-mailx | mailx postgresql-doc postgresql-doc-14 isag
The following NEW packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libtypes-serialiser-perl logrotate netbase
  postgresql postgresql-14 postgresql-client-14 postgresql-client-common postgresql-common
  postgresql-contrib ssl-cert sysstat
0 upgraded, 14 newly installed, 0 to remove and 44 not upgraded.
Need to get 18.4 MB of archives.
After this operation, 52.0 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 logrotate amd64 3.19.0-1ubuntu1.1 [54.3 kB]
Get:2 http://archive.ubuntu.com/ubu

In [44]:
# import psycopg2
# from psycopg2.extras import execute_values
# import time

# N = 10_000_000  # 10 million rows

# # Connect to PostgreSQL
# conn = psycopg2.connect(dbname="testdb", user="postgres", password="postgres", host="localhost")
# cur = conn.cursor()

# # Drop and recreate table to avoid duplication
# cur.execute("DROP TABLE IF EXISTS numbers")
# cur.execute("""
#     CREATE TABLE numbers (
#         id SERIAL PRIMARY KEY,
#         value INTEGER NOT NULL
#     );
# """)
# conn.commit()

# # Insert data in batches
# BATCH_SIZE = 100_000
# start = time.time()

# for i in range(0, N, BATCH_SIZE):
#     batch = [(j % 1000,) for j in range(i, i + BATCH_SIZE)]
#     execute_values(cur, "INSERT INTO numbers (value) VALUES %s", batch)
#     conn.commit()
#     print(f"Inserted {i + BATCH_SIZE} / {N}")

# print(f"✅ Done inserting 10 million rows in {time.time() - start:.2f} seconds.")

# Fetch data to write to input.txt
print("Fetching data to write to input.txt...")
cur.execute("SELECT value FROM numbers")
data = [str(row[0]) for row in cur.fetchall()]
cur.close()
conn.close()

# Save to input.txt for CUDA
with open("input.txt", "w") as f:
    f.write(f"{len(data)}\n")
    f.write('\n'.join(data))

print("📁 input.txt generated for CUDA.")

Fetching data to write to input.txt...
📁 input.txt generated for CUDA.


In [69]:
import psycopg2
import time

# Connect to your local PostgreSQL server
conn = psycopg2.connect(
    dbname="testdb",
    user="postgres",
    password="postgres",
    host="localhost"
)

cursor = conn.cursor()

# SQL aggregation benchmark
start_time = time.time()

cursor.execute("""
    SELECT MAX(value), MIN(value), SUM(value), AVG(value), COUNT(*)
    FROM numbers
""")
max_val, min_val, total_sum, avg, count = cursor.fetchone()

sql_time = time.time() - start_time

print("--- SQL Results ---")
print(f"Maximum: {max_val}")
print(f"Minimum: {min_val}")
print(f"Sum: {total_sum}")
print(f"Average: {avg}")
print(f"Count: {count}")
print(f"SQL Time taken: {sql_time:.6f} s")

cursor.close()
conn.close()

--- SQL Results ---
Maximum: 999
Minimum: 0
Sum: 4995000000
Average: 499.5000000000000000
Count: 10000000
SQL Time taken: 1.212303 s


In [65]:
# Save to input.txt in expected format
with open("input.txt", "w") as f:
    f.write(f"{len(data)}\n")
    f.write(' '.join(map(str, data)))

print("✅ input.txt written successfully.")

✅ input.txt written successfully.


In [66]:
cuda_code = r'''
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>
#include <ctime>
#include <cuda_runtime.h>

#define BLOCK 256

__global__ void find_max_min(const int *arr, int *max_arr, int *min_arr, long long int *sum_arr, long long int *count_arr, int n) {
    __shared__ int shared_max[BLOCK], shared_min[BLOCK];
    __shared__ long long int shared_sum[BLOCK], shared_count[BLOCK];

    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        int val = arr[idx];
        shared_max[tid] = val;
        shared_min[tid] = val;
        shared_sum[tid] = val;
        shared_count[tid] = 1;
    } else {
        shared_max[tid] = INT_MIN;
        shared_min[tid] = INT_MAX;
        shared_sum[tid] = 0;
        shared_count[tid] = 0;
    }
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride /= 2) {
        if (tid < stride) {
            shared_max[tid] = max(shared_max[tid], shared_max[tid + stride]);
            shared_min[tid] = min(shared_min[tid], shared_min[tid + stride]);
            shared_sum[tid] += shared_sum[tid + stride];
            shared_count[tid] += shared_count[tid + stride];
        }
        __syncthreads();
    }

    if (tid == 0) {
        max_arr[blockIdx.x] = shared_max[0];
        min_arr[blockIdx.x] = shared_min[0];
        sum_arr[blockIdx.x] = shared_sum[0];
        count_arr[blockIdx.x] = shared_count[0];
    }
}

int cpu_gpu_aggregate(const int *arr, int n, int *max_val, int *min_val, long long int *sum, float *average, long long int *count, float *gpu_time_ms) {
    int *d_arr, *d_max, *d_min;
    long long int *d_sum, *d_count;

    int num_blocks = (n + BLOCK - 1) / BLOCK;

    int *max_arr = (int *)malloc(sizeof(int) * num_blocks);
    int *min_arr = (int *)malloc(sizeof(int) * num_blocks);
    long long int *sum_arr = (long long int *)malloc(sizeof(long long int) * num_blocks);
    long long int *count_arr = (long long int *)malloc(sizeof(long long int) * num_blocks);

    cudaMalloc(&d_arr, sizeof(int) * n);
    cudaMalloc(&d_max, sizeof(int) * num_blocks);
    cudaMalloc(&d_min, sizeof(int) * num_blocks);
    cudaMalloc(&d_sum, sizeof(long long int) * num_blocks);
    cudaMalloc(&d_count, sizeof(long long int) * num_blocks);

    cudaMemcpy(d_arr, arr, sizeof(int) * n, cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    find_max_min<<<num_blocks, BLOCK>>>(d_arr, d_max, d_min, d_sum, d_count, n);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(gpu_time_ms, start, stop);

    cudaMemcpy(max_arr, d_max, sizeof(int) * num_blocks, cudaMemcpyDeviceToHost);
    cudaMemcpy(min_arr, d_min, sizeof(int) * num_blocks, cudaMemcpyDeviceToHost);
    cudaMemcpy(sum_arr, d_sum, sizeof(long long int) * num_blocks, cudaMemcpyDeviceToHost);
    cudaMemcpy(count_arr, d_count, sizeof(long long int) * num_blocks, cudaMemcpyDeviceToHost);

    *max_val = max_arr[0];
    *min_val = min_arr[0];
    *sum = sum_arr[0];
    *count = count_arr[0];
    for (int i = 1; i < num_blocks; i++) {
        if (max_arr[i] > *max_val) *max_val = max_arr[i];
        if (min_arr[i] < *min_val) *min_val = min_arr[i];
        *sum += sum_arr[i];
        *count += count_arr[i];
    }

    *average = *sum / *count;

    cudaFree(d_arr);
    cudaFree(d_max);
    cudaFree(d_min);
    cudaFree(d_sum);
    cudaFree(d_count);
    free(max_arr);
    free(min_arr);
    free(sum_arr);
    free(count_arr);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

void cpu_aggregate(const int *arr, int n, int *max_val, int *min_val, long long int *sum, float *average, long long int *count, double *cpu_time_ms) {
    clock_t start = clock();

    *max_val = arr[0];
    *min_val = arr[0];
    *sum = arr[0];
    *count = 1;

    for (int i = 1; i < n; i++) {
        if (arr[i] > *max_val) *max_val = arr[i];
        if (arr[i] < *min_val) *min_val = arr[i];
        *sum += arr[i];
        (*count)++;
    }

    *average = *sum / *count;

    clock_t end = clock();
    *cpu_time_ms = (double)(end - start) * 1000.0 / CLOCKS_PER_SEC;
}

int main() {
    FILE *file = fopen("input.txt", "r");
    int n;
    fscanf(file, "%d", &n);
    int *arr = (int *)malloc(sizeof(int) * n);
    for (int i = 0; i < n; i++) {
        fscanf(file, "%d", &arr[i]);
    }
    fclose(file);

    int max_gpu, min_gpu;
    long long int sum_gpu, count_gpu;
    float gpu_time, avg_gpu;
    cpu_gpu_aggregate(arr, n, &max_gpu, &min_gpu, &sum_gpu, &avg_gpu, &count_gpu, &gpu_time);

    int max_cpu, min_cpu;
    long long int sum_cpu, count_cpu;
	float avg_cpu;
    double cpu_time;
    cpu_aggregate(arr, n, &max_cpu, &min_cpu, &sum_cpu, &avg_cpu, &count_cpu, &cpu_time);

    printf("\n--- GPU Results ---\n");
    printf("Maximum: %d\n", max_gpu);
    printf("Minimum: %d\n", min_gpu);
    printf("Sum: %lld\n", sum_gpu);
    printf("Average: %.4f\n", avg_gpu);
    printf("Count: %lld\n", count_gpu);
    printf("GPU Time taken: %.4f ms\n", gpu_time);

    printf("\n--- CPU Results ---\n");
    printf("Maximum: %d\n", max_cpu);
    printf("Minimum: %d\n", min_cpu);
    printf("Sum: %lld\n", sum_cpu);
    printf("Average: %.4f\n", avg_cpu);
    printf("Count: %lld\n", count_cpu);
    printf("CPU Time taken: %.4f ms\n", cpu_time);

    free(arr);
    return 0;
}
'''

# Write to file
with open("cuda_aggregate.cu", "w") as f:
    f.write(cuda_code)

In [67]:
!nvcc -o cuda_aggregate cuda_aggregate.cu

In [68]:
!./cuda_aggregate


--- GPU Results ---
Maximum: 999
Minimum: 0
Sum: 4995000000
Average: 499.0000
Count: 10000000
GPU Time taken: 1.5744 ms

--- CPU Results ---
Maximum: 999
Minimum: 0
Sum: 4995000000
Average: 499.0000
Count: 10000000
CPU Time taken: 38.4040 ms
